# Functional Transforms in PyTorch & DPSGD

In [89]:
import math
import time
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


import torchvision
import torchvision.transforms as transforms


In [90]:
def get_mnist_loader(cfg):
    transform = transforms.Compose([transforms.ToTensor()])
    dataset = torchvision.datasets.MNIST(
        root="/Users/sinngamkhaidem/Developer/torch-jit/data", train=True, download=False, transform=transform
    )
    pin = cfg.device == "cuda"
    return torch.utils.data.DataLoader(
        dataset, batch_size=cfg.batch_size, shuffle=True, drop_last=True, pin_memory=pin
    )

In [91]:
@dataclass
class TrainConfig:
    batch_size: int = 128
    lr: float = 0.1
    epochs: int = 1
    max_grad_norm: float = 1.0
    noise_multiplier: float = 1.1
    vmap_chunk_size: Optional[int] = None
    force_compile_mps: bool = True
    model_name: str = "cnn"  # "cnn" or "mlp"
    jax_data_backend: str = "tfds"  # "tfds" or "torch"
    device: str = (
        "mps"
        if torch.backends.mps.is_available()
        else ("cuda" if torch.cuda.is_available() else "cpu")
    )

cfg = TrainConfig()
if cfg.device == "mps" and cfg.vmap_chunk_size is None:
    cfg.vmap_chunk_size = 32
cfg

TrainConfig(batch_size=128, lr=0.1, epochs=1, max_grad_norm=1.0, noise_multiplier=1.1, vmap_chunk_size=32, force_compile_mps=True, model_name='cnn', jax_data_backend='tfds', device='mps')

In [108]:
class MLP(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

model = MLP().to(cfg.device)
model

MLP(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=10, bias=True)
)

## Per-sample gradient Computation

Gradient of the loss function w.r.t the parameters for a single sample.

In [109]:
from torch.func import functional_call, grad, vmap

def loss_fn(params, buffers, x, y):
    # vmap passes single samples; add batch dim for the model
    x = x.unsqueeze(0)
    y = y.unsqueeze(0)
    logits = functional_call(model, (params, buffers), (x,))
    return F.cross_entropy(logits, y)

def make_per_sample_grads(model, chunk_size=None):
    params = {k: v for k, v in model.named_parameters()}
    buffers = {k: v for k, v in model.named_buffers()}
    grad_fn = grad(loss_fn)
    compute_per_sample_grads = vmap(grad_fn, in_dims=(None, None, 0, 0), chunk_size=chunk_size)
    return params, buffers, compute_per_sample_grads

params, buffers, compute_per_sample_grads = make_per_sample_grads(model, cfg.vmap_chunk_size)
param_names = list(params.keys())
len(params)

6

In [139]:
x, y = next(iter(get_mnist_loader(cfg)))
x, y = x.to(cfg.device), y.to(cfg.device)
print(x.shape, y.shape)

torch.Size([128, 1, 28, 28]) torch.Size([128])


In [140]:
per_sample_grads = compute_per_sample_grads(params, buffers, x, y)

In [117]:
def clip_and_aggregate(per_sample_grads, max_grad_norm, noise_multiplier, batch_size, device, param_names):
    grads = [per_sample_grads[name] for name in param_names]

    sq_norms = torch.stack([g.flatten(1).pow(2).sum(1) for g in grads]).sum(0)
    norms = torch.sqrt(sq_norms + 1e-12)

    # Clip factors per sample
    clip_factor = torch.clamp(max_grad_norm / norms, max=1.0)

    # Apply clipping and aggregate
    clipped = []
    for g in grads:
        # Reshape clip_factor to broadcast correctly with per-sample grads
        factor = clip_factor.view(-1, *([1] * (g.dim() - 1))) # clip_factor.view(-1, 1, 1)
        clipped.append((g * factor).mean(0))

    noise_std = noise_multiplier * max_grad_norm / batch_size
    device_type = device.type if isinstance(device, torch.device) else str(device)
    use_foreach = device_type != "mps" and hasattr(torch, "_foreach_add")
    if noise_std > 0:
        if use_foreach:
            noise = [torch.randn_like(g, device=device) * noise_std for g in clipped]
            noisy = torch._foreach_add(clipped, noise)
        else:
            noisy = [g + torch.randn_like(g, device=device) * noise_std for g in clipped]
    else:
        noisy = clipped

    return dict(zip(param_names, noisy))

In [ ]:
# TEST clip_and_aggregate
per_sample_grads = compute_per_sample_grads(params, buffers, x, y)
noisy_grads = clip_and_aggregate(
    per_sample_grads,
    cfg.max_grad_norm,
    cfg.noise_multiplier,
    x.shape[0],
    cfg.device,
    param_names,
)


In [119]:
def dp_sgd_step(params, buffers, batch, cfg):
    x, y = batch
    per_sample_grads = compute_per_sample_grads(params, buffers, x, y)
    noisy_grads = clip_and_aggregate(
        per_sample_grads,
        cfg.max_grad_norm,
        cfg.noise_multiplier,
        x.shape[0],
        x.device,
        param_names,
    )
    # SGD update using foreach for lower overhead (skip on MPS)
    params_list = [params[name] for name in param_names]
    grads_list = [noisy_grads[name] for name in param_names]
    
    device_type = x.device.type
    use_foreach = device_type != "mps" and hasattr(torch, "_foreach_add")

    if use_foreach:
        new_params_list = torch._foreach_add(params_list, grads_list, alpha=-cfg.lr)
    else:
        new_params_list = [p - cfg.lr * g for p, g in zip(params_list, grads_list)]
    return dict(zip(param_names, new_params_list))

## Step 6: Compiled training step
Wraps the DP-SGD update with `torch.compile()`.

The compiler can fuse operations into optimized kernels, reducing Python overhead.

**Difference from Opacus**
- Opacus hooks prevent full-graph compilation because they run in Python.
- Here the step is a pure tensor function, enabling compilation.

**Difference from JAX**
- JAX compiles by default and stages the whole update into XLA.
- `torch.compile()` is the PyTorch analog, but success depends on avoiding dynamic control flow.

In [120]:
def make_compiled_step():
    if hasattr(torch, "compile"):
        if cfg.device == "mps" and not cfg.force_compile_mps:
            # MPS compile support is limited; eager is often faster and more stable
            return dp_sgd_step
        
        else:
            print("Using torch.compile for dp_sgd_step.")
            return torch.compile(
                dp_sgd_step,
                fullgraph=True,
                dynamic=False,
                mode="reduce-overhead",
            )
    return dp_sgd_step

compiled_step = make_compiled_step()

Using torch.compile for dp_sgd_step.


## Step 6b: Compile diagnostics
Use `torch._dynamo.explain` to report graph breaks and compilation coverage.

Shows which Python/control-flow ops prevent fusion and explains why compile may not help.

In [121]:
def explain_compile(cfg):
    if not hasattr(torch, "_dynamo"):
        print("torch._dynamo is not available")
        return
    if cfg.device == "mps" and not cfg.force_compile_mps:
        print("MPS compile disabled; set cfg.force_compile_mps = True to analyze.")
        return
    # Build a tiny batch to trace
    loader = get_mnist_loader(cfg)
    x, y = next(iter(loader))
    x = x.to(cfg.device)
    y = y.to(cfg.device)
    params = {k: v.detach() for k, v in model.named_parameters()}
    buffers = {k: v for k, v in model.named_buffers()}
    
    report = torch._dynamo.explain(dp_sgd_step, params, buffers, (x, y), cfg)
    print(report)

# Uncomment to run diagnostics
explain_compile(cfg)

Graph Count: 1
Graph Break Count: 0
Op Count: 300
Break Reasons:
Ops per Graph:
  Ops 1:
    <function lazy_load_decompositions at 0x117d11000>
    <built-in function getitem>
    <built-in function getitem>
    <built-in function getitem>
    <built-in function getitem>
    <built-in function getitem>
    <built-in function getitem>
    <built-in function getitem>
    <built-in function getitem>
    <function _vmap_increment_nesting at 0x117d10ee0>
    <function _add_batch_dim at 0x117d10dc0>
    <function _add_batch_dim at 0x117d10dc0>
    <built-in method _saved_tensors_hooks_disable of pybind11_builtins.pybind11_detail_function_record_v1_system_libcpp_abi1 object at 0x117ca1d50>
    <built-in method _grad_increment_nesting of pybind11_builtins.pybind11_detail_function_record_v1_system_libcpp_abi1 object at 0x117018c50>
    <built-in method _wrap_for_grad of pybind11_builtins.pybind11_detail_function_record_v1_system_libcpp_abi1 object at 0x117018cd0>
    <built-in method _wrap_for_

## Step 7: Minimal training loop

You can swap this for CIFAR or IMDb by changing the dataset and model.

In [122]:
def _sync_device(device):
    if device == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize()
    elif device == "mps" and hasattr(torch, "mps") and torch.backends.mps.is_available():
        torch.mps.synchronize()

def train_one_epoch(cfg, model):
    loader = get_mnist_loader(cfg)

    params = {k: v.detach() for k, v in model.named_parameters()}
    buffers = {k: v for k, v in model.named_buffers()}

    model.train()

    _sync_device(cfg.device)
    start = time.perf_counter()
    non_blocking = cfg.device == "cuda"
    for x, y in loader:
        x = x.to(cfg.device, non_blocking=non_blocking)
        y = y.to(cfg.device, non_blocking=non_blocking)
        params = compiled_step(params, buffers, (x, y), cfg)
    _sync_device(cfg.device)

    elapsed = time.perf_counter() - start
    return elapsed

train_time = train_one_epoch(cfg, model)
print(f"Epoch time: {train_time:.2f}s")

Epoch time: 4.96s


## Step 8: Benchmarking protocol

**Recommended runs**
- Non-private SGD (baseline)
- DP-SGD without compile (same functional path)
- DP-SGD with compile (this notebook)
- Opacus DP-SGD (hook-based)
- JAX DP-SGD (from prior work, via reported numbers)

## JAX 1-epoch comparison
Runs a JAX DP-SGD step that mirrors the paper's JIT+vmap implementation (Haiku + vmap + jit).

This adapts the paper logic for MNIST and replaces the external data pipeline with the local PyTorch DataLoader.

In [123]:
import numpy as np

try:
    import haiku as hk
    import jax
    import jax.numpy as jnp
    from jax import grad, jit, random, vmap
    from jax.tree_util import tree_flatten, tree_unflatten
    try:
        import tensorflow as tf
        import tensorflow_datasets as tfds
        _HAS_TFDS = True
    except Exception:
        _HAS_TFDS = False
except Exception as exc:
    raise RuntimeError("JAX + Haiku are required for this section. Install jax, jaxlib, and dm-haiku.") from exc

def mnist_cnn(features):
    return hk.Sequential([
        hk.Conv2D(16, (8, 8), padding="SAME", stride=(2, 2)),
        jax.nn.relu,
        hk.MaxPool(2, 1, padding="VALID"),
        hk.Conv2D(32, (4, 4), padding="VALID", stride=(2, 2)),
        jax.nn.relu,
        hk.MaxPool(2, 1, padding="VALID"),
        hk.Flatten(),
        hk.Linear(32),
        jax.nn.relu,
        hk.Linear(10),
    ])(features)

def mnist_mlp(features):
    x = hk.Flatten()(features)
    x = hk.Linear(256)(x)
    x = jax.nn.relu(x)
    x = hk.Linear(128)(x)
    x = jax.nn.relu(x)
    return hk.Linear(10)(x)

def build_jax_model(cfg):
    if cfg.model_name == "mlp":
        return hk.transform(mnist_mlp)
    return hk.transform(mnist_cnn)

jax_model = build_jax_model(cfg)

def multiclass_loss(model, params, batch):
    inputs, targets = batch
    logits = model.apply(params, None, inputs)
    one_hot = jax.nn.one_hot(targets, logits.shape[-1])
    log_probs = jax.nn.log_softmax(logits)
    return -jnp.mean(jnp.sum(log_probs * one_hot, axis=-1))

def clipped_grad(model, loss, params, l2_norm_clip, single_example_batch):
    inputs, targets = single_example_batch
    if inputs.ndim == 3:
        # Ensure a batch dimension for Haiku modules
        inputs = inputs[None, ...]
        targets = jnp.reshape(targets, (1,))
    grads = grad(lambda p: loss(model, p, (inputs, targets)))(params)
    nonempty_grads, tree_def = tree_flatten(grads)
    per_param = jnp.stack([jnp.linalg.norm(g.ravel()) for g in nonempty_grads])
    total_norm = jnp.linalg.norm(per_param)
    divisor = jnp.maximum(total_norm / l2_norm_clip, 1.0)
    normalized = [g / divisor for g in nonempty_grads]
    return tree_unflatten(tree_def, normalized)

def private_grad_vmap(model, loss, params, batch, rng, l2_norm_clip, noise_multiplier, batch_size):
    clipped = vmap(lambda eg: clipped_grad(model, loss, params, l2_norm_clip, eg))(batch)
    clipped_flat, treedef = tree_flatten(clipped)
    aggregated = [g.sum(0) for g in clipped_flat]
    rngs = random.split(rng, len(aggregated))
    noised = [g + l2_norm_clip * noise_multiplier * random.normal(r, g.shape) for r, g in zip(rngs, aggregated)]
    normalized = [g / batch_size for g in noised]
    return tree_unflatten(treedef, normalized)

def private_grad_no_vmap(model, loss, params, batch, rng, l2_norm_clip, noise_multiplier, batch_size):
    grads = [clipped_grad(model, loss, params, l2_norm_clip, eg) for eg in batch]
    grads_flat = [tree_flatten(g)[0] for g in grads]
    treedef = tree_flatten(grads[0])[1]
    stacked = [jnp.stack([g[i] for g in grads_flat]) for i in range(len(grads_flat[0]))]
    aggregated = [g.sum(0) for g in stacked]
    rngs = random.split(rng, len(aggregated))
    noised = [g + l2_norm_clip * noise_multiplier * random.normal(r, g.shape) for r, g in zip(rngs, aggregated)]
    normalized = [g / batch_size for g in noised]
    return tree_unflatten(treedef, normalized)

def dp_sgd_step_jax_vmap(params, batch, rng, lr, max_grad_norm, noise_multiplier, batch_size):
    grads = private_grad_vmap(
        jax_model, multiclass_loss, params, batch, rng, max_grad_norm, noise_multiplier, batch_size
    )
    return jax.tree_util.tree_map(lambda p, g: p - lr * g, params, grads)

def dp_sgd_step_jax_no_vmap(params, batch, rng, lr, max_grad_norm, noise_multiplier, batch_size):
    grads = private_grad_no_vmap(
        jax_model, multiclass_loss, params, batch, rng, max_grad_norm, noise_multiplier, batch_size
    )
    return jax.tree_util.tree_map(lambda p, g: p - lr * g, params, grads)

def _to_nhwc(x):
    if x.ndim != 4:
        raise ValueError(f"Expected 4D input, got shape {x.shape}")
    if x.shape[1] == 1:
        x = np.transpose(x, (0, 2, 3, 1))
    elif x.shape[-1] != 1:
        raise ValueError(f"Expected channel=1, got shape {x.shape}")
    if x.shape[1] != 28 or x.shape[2] != 28:
        raise ValueError(f"Expected 28x28 input, got shape {x.shape}")
    return x.astype(np.float32, copy=False)

def _prepare_jax_batch(x, y):
    if hasattr(x, "numpy"):
        x = x.numpy()
    if hasattr(y, "numpy"):
        y = y.numpy()
    x = _to_nhwc(x)
    y = y.astype(np.int64, copy=False)
    return x, y

def get_mnist_loader_jax(cfg):
    if cfg.jax_data_backend != "tfds" or not _HAS_TFDS:
        return None
    ds = tfds.load("mnist", split="train", as_supervised=True)
    ds = ds.map(
        lambda x, y: (tf.cast(x, tf.float32) / 255.0, tf.cast(y, tf.int64)),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    ds = ds.batch(cfg.batch_size, drop_remainder=False)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return tfds.as_numpy(ds)

def train_one_epoch_jax(cfg, use_vmap=True, use_jit=True):
    # Prefer TFDS to avoid torch->numpy transfer overhead.
    loader = get_mnist_loader_jax(cfg)
    if loader is None:
        loader = get_mnist_loader(cfg)

    rng = random.PRNGKey(0)

    sample_x, sample_y = next(iter(loader))
    sample_x, sample_y = _prepare_jax_batch(sample_x, sample_y)

    params = jax_model.init(rng, sample_x)

    step = dp_sgd_step_jax_vmap if use_vmap else dp_sgd_step_jax_no_vmap
    
    if use_jit:
        step = jit(step)
        
    start = time.perf_counter()
    for x, y in loader:
        x, y = _prepare_jax_batch(x, y)
        x = jax.device_put(x)
        y = jax.device_put(y)
        batch = (x, y)
        params = step(
            params, batch, rng, cfg.lr, cfg.max_grad_norm, cfg.noise_multiplier, x.shape[0]
        )
    jax.block_until_ready(params)
    return time.perf_counter() - start

In [124]:
jax_time = train_one_epoch_jax(cfg)

print(f"JAX epoch time: {jax_time:.2f}s")

if "train_time" in globals():
    print(f"PyTorch epoch time: {train_time:.2f}s")
    ratio = train_time / jax_time if jax_time > 0 else float("inf")
    print(f"PyTorch/JAX ratio: {ratio:.2f}x (lower is faster)")
else:
    print("PyTorch epoch time not found; run the PyTorch epoch cell first.")

JAX epoch time: 5.67s
PyTorch epoch time: 4.96s
PyTorch/JAX ratio: 0.87x (lower is faster)


## Opacus 1-epoch comparison
Runs Opacus DP-SGD for one epoch on the same MNIST loader.

This uses `PrivacyEngine.make_private` with the same `noise_multiplier` and `max_grad_norm` as the compiled path.

In [125]:
from opacus import PrivacyEngine

def train_one_epoch_opacus(cfg, grad_sample_mode="hooks"):
    loader = get_mnist_loader(cfg)

    model = build_model(cfg).to(cfg.device)
    optimizer = torch.optim.SGD(model.parameters(), lr=cfg.lr)
    privacy_engine = PrivacyEngine()

    model, optimizer, private_loader = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=loader,
        noise_multiplier=cfg.noise_multiplier,
        max_grad_norm=cfg.max_grad_norm,
        grad_sample_mode=grad_sample_mode
        )

    criterion = nn.CrossEntropyLoss(reduction="mean")
    model.train()
    start = time.perf_counter()
    for x, y in private_loader:
        x = x.to(cfg.device)
        y = y.to(cfg.device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
    return time.perf_counter() - start

opacus_time = train_one_epoch_opacus(cfg, "functorch")
print(f"Opacus epoch time: {opacus_time:.2f}s")
if "train_time" in globals():
    print(f"PyTorch epoch time: {train_time:.2f}s")
    ratio = opacus_time / train_time if train_time > 0 else float("inf")
    print(f"Opacus/PyTorch ratio: {ratio:.2f}x (higher means Opacus is slower)")
else:
    print("PyTorch epoch time not found; run the PyTorch epoch cell first.")

NameError: name 'build_model' is not defined

## Overall comparison
Runs all local baselines and summarizes timings alongside optional JAX reported numbers.

Gives a single place to compare non-private SGD, DP-SGD (functional, compiled, Opacus), and JAX (reported).


In [ ]:
def train_one_epoch_non_private(cfg):
    loader = get_mnist_loader(cfg)
    model = build_model(cfg).to(cfg.device)
    optimizer = torch.optim.SGD(model.parameters(), lr=cfg.lr)
    criterion = nn.CrossEntropyLoss(reduction="mean")
    model.train()
    _sync_device(cfg.device)
    start = time.perf_counter()
    non_blocking = cfg.device == "cuda"
    for x, y in loader:
        x = x.to(cfg.device, non_blocking=non_blocking)
        y = y.to(cfg.device, non_blocking=non_blocking)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
    _sync_device(cfg.device)
    return time.perf_counter() - start

def train_one_epoch_dp_functional(cfg, use_compile):
    loader = get_mnist_loader(cfg)
    model = build_model(cfg).to(cfg.device)

    params = {k: v.detach() for k, v in model.named_parameters()}
    buffers = {k: v for k, v in model.named_buffers()}
    step_fn = compiled_step if use_compile else dp_sgd_step

    model.train()
    _sync_device(cfg.device)
    start = time.perf_counter()
    non_blocking = cfg.device == "cuda"
    for x, y in loader:
        x = x.to(cfg.device, non_blocking=non_blocking)
        y = y.to(cfg.device, non_blocking=non_blocking)
        params = step_fn(params, buffers, (x, y), cfg)
    _sync_device(cfg.device)
    return time.perf_counter() - start

def compare_all(cfg):
    results = {}
    results["Non-private SGD"] = train_one_epoch_non_private(cfg)
    results["DP-SGD functional (no compile)"] = train_one_epoch_dp_functional(cfg, use_compile=False)
    results["DP-SGD functional (compile)"] = train_one_epoch_dp_functional(cfg, use_compile=True)
    try:
        results["Opacus DP-SGD"] = train_one_epoch_opacus(cfg, "ew")
    except Exception as exc:
        results["Opacus DP-SGD"] = f"error: {exc}"
    if "jax_time" in globals():
        results["JAX DP-SGD (this run)"] = jax_time
    jax_reported_time = None  # set to a float (seconds) if you want to include reported numbers
    if isinstance(jax_reported_time, (int, float)):
        results["JAX DP-SGD (reported)"] = float(jax_reported_time)
    return results

results = compare_all(cfg)
for k, v in results.items():
    print(f"{k}: {v}")

04/11/2026 14:32:32:WARNING:Ignoring drop_last as it is not compatible with DPDataLoader.


Non-private SGD: 3.4849767920095474
DP-SGD functional (no compile): 8.7073945000011
DP-SGD functional (compile): 5.074611832998926
Opacus DP-SGD: 10.6656826659746
JAX DP-SGD (this run): 4.859422167006414


## Step 9: Privacy accounting validation
Validates that $"(ε, δ)"$ is computed consistently with standard DP-SGD. Performance gains must not alter privacy guarantees.

This notebook keeps the DP mechanics identical to standard DP-SGD. You can validate the accounting using Opacus or your preferred accountant implementation.

In [ ]:
try:
    from opacus.accountants import RDPAccountant
except Exception as exc:
    raise RuntimeError("Opacus is required for privacy accounting. Install opacus.") from exc

def _compute_epsilon(cfg, delta=1e-5):
    loader = get_mnist_loader(cfg)
    steps = cfg.epochs * len(loader)
    sample_rate = cfg.batch_size / len(loader.dataset)
    accountant = RDPAccountant()
    for _ in range(steps):
        accountant.step(noise_multiplier=cfg.noise_multiplier, sample_rate=sample_rate)
    return accountant.get_epsilon(delta)

def compute_epsilon_opacus(cfg, delta=1e-5):
    return _compute_epsilon(cfg, delta)

def compute_epsilon_jax(cfg, delta=1e-5):
    return _compute_epsilon(cfg, delta)

# Example usage
epsilon_opacus = compute_epsilon_opacus(cfg, delta=1e-5)
epsilon_jax = compute_epsilon_jax(cfg, delta=1e-5)
print(f"Opacus epsilon (delta=1e-5): {epsilon_opacus:.4f}")
print(f"JAX epsilon (delta=1e-5): {epsilon_jax:.4f}")

Opacus epsilon (delta=1e-5): 0.6294
JAX epsilon (delta=1e-5): 0.6294



**Opacus (PyTorch)**
- Per-sample gradients via hooks (Python callbacks).
- User-friendly `PrivacyEngine`, but hooks block full-graph compilation.
- Typically 3-8x slower than non-private training on large models.

**JAX DP-SGD**
- Uses `vmap` + XLA JIT to fuse the entire update.
- Near parity with non-private SGD in published work.
- Requires JAX ecosystem and XLA backend.

**This PyTorch JIT DP-SGD**
- Uses `torch.func.vmap` + `grad` for per-sample gradients.
- Encapsulates DP-SGD in a single tensor function suitable for `torch.compile()`.
- Aims to close the gap with JAX while preserving PyTorch ergonomics.